In [1]:
import pandas as pd
import numpy as np
import nltk
import spacy
import gensim
import warnings
warnings.filterwarnings("ignore")

In [2]:
# DATA IMPORTING 

In [3]:
test_df = pd.read_csv(
    r"C:\Users\Lenovo\Desktop\github projects\nlp rnn assingment\data train test validate\test.txt",
    sep=";",
    names=["text", "emotion"]
)
train_df = pd.read_csv(
    r"C:\Users\Lenovo\Desktop\github projects\nlp rnn assingment\data train test validate\train.txt",
    sep=";",
    names=["text", "emotion"]
)
val_df = pd.read_csv(
    r"C:\Users\Lenovo\Desktop\github projects\nlp rnn assingment\data train test validate\val.txt",
    sep=";",
    names=["text", "emotion"]
)

In [4]:
print(test_df.head())
print("\n")
print(train_df.head())
print("\n")
print(train_df.head())

                                                text  emotion
0  im feeling rather rotten so im not very ambiti...  sadness
1          im updating my blog because i feel shitty  sadness
2  i never make her separate from me because i do...  sadness
3  i left with my bouquet of red and yellow tulip...      joy
4    i was feeling a little vain when i did this one  sadness


                                                text  emotion
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i am ever feeling nostalgic about the fireplac...     love
4                               i am feeling grouchy    anger


                                                text  emotion
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i

In [5]:
 # checking for missing values
for name,df in [("Train",train_df),("Test",test_df),("valadition", val_df)]:
    print(f"\n{name} Dataset")
    print(df.isnull().sum())


Train Dataset
text       0
emotion    0
dtype: int64

Test Dataset
text       0
emotion    0
dtype: int64

valadition Dataset
text       0
emotion    0
dtype: int64


In [6]:
# checking for duplicate values 

In [7]:
for name ,df in [("Train",train_df),("Test",test_df),("validate",val_df)]:
    print(f"\n{name} Dataset")
    print(df.duplicated().sum())


Train Dataset
1

Test Dataset
0

validate Dataset
0


In [8]:
# Basic Preprocessing 
import re 

def clean_text(text):
    text=text.lower()

    text=re.sub(r'http\S+|www\S+','',text)

    text=re.sub(r'<.*?>', '', text)

    text=re.sub(r'[^a-zA-Z\s]', '', text)

    text=re.sub(r'\s+',' ',text).strip()

    return text


In [9]:
train_df["clean_text"]=train_df["text"].apply(clean_text)
test_df["clean_text"]=test_df["text"].apply(clean_text)
val_df["clean_text"]=val_df["text"].apply(clean_text)

In [10]:
for head in [ train_df,test_df,val_df]:
    print(display(head[["text","clean_text"]].head()))
    print("--"*50)

,text,clean_text
0,i didnt feel humiliated,i didnt feel humiliated
1,i can go from feeling so hopeless to so damned...,i can go from feeling so hopeless to so damned...
2,im grabbing a minute to post i feel greedy wrong,im grabbing a minute to post i feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...,i am ever feeling nostalgic about the fireplac...
4,i am feeling grouchy,i am feeling grouchy


None
----------------------------------------------------------------------------------------------------


,text,clean_text
0,im feeling rather rotten so im not very ambiti...,im feeling rather rotten so im not very ambiti...
1,im updating my blog because i feel shitty,im updating my blog because i feel shitty
2,i never make her separate from me because i do...,i never make her separate from me because i do...
3,i left with my bouquet of red and yellow tulip...,i left with my bouquet of red and yellow tulip...
4,i was feeling a little vain when i did this one,i was feeling a little vain when i did this one


None
----------------------------------------------------------------------------------------------------


,text,clean_text
0,im feeling quite sad and sorry for myself but ...,im feeling quite sad and sorry for myself but ...
1,i feel like i am still looking at a blank canv...,i feel like i am still looking at a blank canv...
2,i feel like a faithful servant,i feel like a faithful servant
3,i am just feeling cranky and blue,i am just feeling cranky and blue
4,i can have for a treat or if i am feeling festive,i can have for a treat or if i am feeling festive


None
----------------------------------------------------------------------------------------------------


#### Tokenization 

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
tokenizer=Tokenizer(num_words=10000,oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["clean_text"])

In [13]:
X_train=tokenizer.texts_to_sequences(train_df["clean_text"])
X_test=tokenizer.texts_to_sequences(test_df["clean_text"])
X_val=tokenizer.texts_to_sequences(val_df["clean_text"])

In [14]:
# Label Encododint(Target Variabel)
from sklearn.preprocessing import LabelEncoder

In [15]:
label_encoder=LabelEncoder()

In [16]:
y_train=label_encoder.fit_transform(train_df["emotion"])
y_test=label_encoder.transform(test_df["emotion"])
y_val=label_encoder.transform(val_df["emotion"])

In [17]:
label_encoder.classes_

array(['anger', 'fear', 'joy', 'love', 'sadness', 'surprise'],
      dtype=object)

In [18]:
#### Padding 
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [51]:
max_length=60

X_train=pad_sequences(X_train,maxlen=max_length,padding='post',truncating='post')
X_test=pad_sequences(X_test,maxlen=max_length,padding='post',truncating='post')
X_val=pad_sequences(X_val,maxlen=max_length,padding='post',truncating='post')

# MODELLING

In [40]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding,SimpleRNN,LSTM,GRU,Dense,Dropout,Bidirectional)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (classification_report,confusion_matrix,accuracy_score)

In [21]:
vocab_size=len(tokenizer.word_index)+1

max_length=X_train.shape[1]

num_classes=len(label_encoder.classes_)

embedding_dim=128


# EVALUATION FUNCTION 

In [80]:
def evaluate_model(model,X_train,y_train,X_val,y_val,X_test,y_test):
    # Train Predictions 
    train_pred=model.predict(X_train,verbose=0)
    train_pred_classes=np.argmax(train_pred,axis=1)

    #Test Predictions
    test_pred=model.predict(X_test,verbose=0)
    test_pred_classes=np.argmax(test_pred,axis=1)

    #Validation_Prediction
    val_pred = model.predict(X_val, verbose=0)
    val_pred_classes = np.argmax(val_pred, axis=1)

    #ACCURACY
    train_acc=accuracy_score(y_train,train_pred_classes)
    val_acc = accuracy_score(y_val, val_pred_classes)
    test_acc=accuracy_score(y_test,test_pred_classes)

    #cofusion Matrix 
    cm=confusion_matrix(y_test,test_pred_classes)
    #classification_report 
    report=classification_report(y_test,test_pred_classes)

    print("MODEL EVALUATION")
    print("="*50)

    print(f"\n Train Accuracy :{train_acc}")
    print(f"\n Validition Accuracy :{val_acc}")
    print(f"\n Test Accuracy :{test_acc}")

    print("\n Classification Report")
    print(report)

    print("\n Confusion Matrix")
    print(cm)
    
    
    
    



### 1.Simple RNN

In [23]:
rnn_model=Sequential([Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_length),
                      SimpleRNN(128),Dense(num_classes,activation='softmax')])

rnn_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
rnn_model.build(input_shape=(None, max_length))

rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 60, 128)        │     1,947,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,980,678 (7.56 MB)

 Trainable params: 1,980,678 (7.56 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
history_rnn=rnn_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 39ms/step - accuracy: 0.3205 - loss: 1.5905 - val_accuracy: 0.3515 - val_loss: 1.5832
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.3349 - loss: 1.5807 - val_accuracy: 0.3515 - val_loss: 1.5840
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.3283 - loss: 1.5812 - val_accuracy: 0.3500 - val_loss: 1.5875
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.3284 - loss: 1.5782 - val_accuracy: 0.3510 - val_loss: 1.5836
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.3334 - loss: 1.5761 - val_accuracy: 0.3505 - val_loss: 1.5892
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.3338 - loss: 1.5751 - val_accuracy: 0.3520 - val_loss: 1.5849
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 37ms/step - accuracy: 0.3332 - loss: 1.5810 - val_accuracy: 0.2750 - val_loss: 1.7189
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.3198 - loss: 1.5969 - 

In [82]:
evaluate_model(rnn_model,X_train,y_train,X_val,y_val,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.335125

 Validition Accuracy :0.3515

 Test Accuracy :0.3475

 Classification Report
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       275
           1       0.00      0.00      0.00       224
           2       0.35      1.00      0.52       695
           3       0.00      0.00      0.00       159
           4       0.00      0.00      0.00       581
           5       0.00      0.00      0.00        66

    accuracy                           0.35      2000
   macro avg       0.06      0.17      0.09      2000
weighted avg       0.12      0.35      0.18      2000


 Confusion Matrix
[[  0   0 275   0   0   0]
 [  0   0 224   0   0   0]
 [  0   0 695   0   0   0]
 [  0   0 159   0   0   0]
 [  0   0 581   0   0   0]
 [  0   0  66   0   0   0]]


### 2. LSTM

In [26]:
lstm_model=Sequential([Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_length),LSTM(128),
                       Dense(num_classes,activation='softmax')])
lstm_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
lstm_model.build(input_shape=(None, max_length))
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 60, 128)        │     1,947,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,079,366 (7.93 MB)

 Trainable params: 2,079,366 (7.93 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
history_lstm=lstm_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 65ms/step - accuracy: 0.3288 - loss: 1.5829 - val_accuracy: 0.3520 - val_loss: 1.5827
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 32s 65ms/step - accuracy: 0.3316 - loss: 1.5777 - val_accuracy: 0.3520 - val_loss: 1.5809
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 69ms/step - accuracy: 0.3332 - loss: 1.5765 - val_accuracy: 0.3520 - val_loss: 1.5818
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 65ms/step - accuracy: 0.3292 - loss: 1.5765 - val_accuracy: 0.2755 - val_loss: 1.5878
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 34s 67ms/step - accuracy: 0.3936 - loss: 1.3672 - val_accuracy: 0.5110 - val_loss: 1.1029
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 36s 71ms/step - accuracy: 0.6961 - loss: 0.7783 - val_accuracy: 0.8575 - val_loss: 0.5411
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 37s 75ms/step - accuracy: 0.8857 - loss: 0.3989 - val_accuracy: 0.8860 - val_loss: 0.3855
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 37s 74ms/step - accuracy: 0.9156 - loss: 0.2666 - 

In [83]:
evaluate_model(lstm_model,X_train,y_train,X_val,y_val,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.954

 Validition Accuracy :0.9075

 Test Accuracy :0.906

 Classification Report
              precision    recall  f1-score   support

           0       0.93      0.91      0.92       275
           1       0.89      0.87      0.88       224
           2       0.91      0.92      0.92       695
           3       0.74      0.86      0.80       159
           4       0.96      0.95      0.95       581
           5       0.74      0.61      0.67        66

    accuracy                           0.91      2000
   macro avg       0.86      0.85      0.86      2000
weighted avg       0.91      0.91      0.91      2000


 Confusion Matrix
[[249   7  10   0   9   0]
 [  4 195   5   0   7  13]
 [  2   2 638  47   5   1]
 [  2   0  18 137   2   0]
 [ 10   4  14   0 553   0]
 [  0  10  14   0   2  40]]


### 3. GRU

In [33]:
gru_model=Sequential([Embedding(vocab_size,
              embedding_dim,
              input_length=max_length),

    GRU(128),

    Dense(num_classes,
          activation='softmax')
])
gru_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])



In [34]:
history_gru=gru_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 30s 55ms/step - accuracy: 0.3273 - loss: 1.5845 - val_accuracy: 0.3520 - val_loss: 1.5903
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 28s 56ms/step - accuracy: 0.3264 - loss: 1.5793 - val_accuracy: 0.3520 - val_loss: 1.5849
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - accuracy: 0.4733 - loss: 1.3002 - val_accuracy: 0.8390 - val_loss: 0.4783
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - accuracy: 0.9122 - loss: 0.2535 - val_accuracy: 0.9265 - val_loss: 0.1825
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 29s 58ms/step - accuracy: 0.9461 - loss: 0.1241 - val_accuracy: 0.9300 - val_loss: 0.1584
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 29s 58ms/step - accuracy: 0.9589 - loss: 0.0897 - val_accuracy: 0.9315 - val_loss: 0.1657
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 65ms/step - accuracy: 0.9675 - loss: 0.0709 - val_accuracy: 0.9305 - val_loss: 0.1717
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 29s 59ms/step - accuracy: 0.9753 - loss: 0.0551 - 

In [84]:
evaluate_model(gru_model,X_train,y_train,X_val,y_val,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.990625

 Validition Accuracy :0.9335

 Test Accuracy :0.925

 Classification Report
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       275
           1       0.86      0.92      0.89       224
           2       0.95      0.95      0.95       695
           3       0.86      0.81      0.83       159
           4       0.96      0.96      0.96       581
           5       0.74      0.61      0.67        66

    accuracy                           0.93      2000
   macro avg       0.88      0.86      0.87      2000
weighted avg       0.92      0.93      0.92      2000


 Confusion Matrix
[[256   7   5   0   7   0]
 [  4 207   1   0   7   5]
 [  3   3 658  21   4   6]
 [  1   2  24 129   1   2]
 [ 12   4   4   0 560   1]
 [  0  17   3   0   6  40]]


### 4. Bi-Directional RNN

In [43]:
birnn_model=Sequential([Embedding(vocab_size,embedding_dim,input_length=max_length),Bidirectional(SimpleRNN(128)),
                       Dense(num_classes,activation='softmax')])

birnn_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

                                                                                                 

In [44]:
history_birnn=birnn_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 24s 40ms/step - accuracy: 0.4028 - loss: 1.4827 - val_accuracy: 0.5650 - val_loss: 1.2263
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - accuracy: 0.7143 - loss: 0.8067 - val_accuracy: 0.7410 - val_loss: 0.7308
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.8736 - loss: 0.3765 - val_accuracy: 0.7550 - val_loss: 0.7725
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.9387 - loss: 0.1924 - val_accuracy: 0.7935 - val_loss: 0.6786
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - accuracy: 0.9435 - loss: 0.1806 - val_accuracy: 0.7900 - val_loss: 0.7567
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 40ms/step - accuracy: 0.9734 - loss: 0.0873 - val_accuracy: 0.8040 - val_loss: 0.7604
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - accuracy: 0.9833 - loss: 0.0558 - val_accuracy: 0.7965 - val_loss: 0.8145
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 40ms/step - accuracy: 0.9582 - loss: 0.1304 - 

In [85]:
evaluate_model(birnn_model,X_train,y_train,X_val,y_val,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.981625

 Validition Accuracy :0.766

 Test Accuracy :0.753

 Classification Report
              precision    recall  f1-score   support

           0       0.76      0.74      0.75       275
           1       0.77      0.61      0.68       224
           2       0.81      0.83      0.82       695
           3       0.44      0.69      0.54       159
           4       0.86      0.80      0.83       581
           5       0.28      0.23      0.25        66

    accuracy                           0.75      2000
   macro avg       0.65      0.65      0.65      2000
weighted avg       0.77      0.75      0.76      2000


 Confusion Matrix
[[204  18  11   9  32   1]
 [ 25 137  14  16  14  18]
 [  4   0 578  80  26   7]
 [  4   1  29 110   4  11]
 [ 31   9  70   8 462   1]
 [  0  13  13  25   0  15]]


1

### 5. BI-Directional LSTM

In [56]:
bilstm_model = Sequential([

    Embedding(vocab_size,
              embedding_dim,
              input_length=max_length),

    Bidirectional(
        LSTM(128)
    ),

    Dense(num_classes,
          activation='softmax')
])

In [57]:
bilstm_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [58]:
history_bilstm=bilstm_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 38s 70ms/step - accuracy: 0.6373 - loss: 0.9914 - val_accuracy: 0.8905 - val_loss: 0.3440
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 47s 93ms/step - accuracy: 0.9164 - loss: 0.2358 - val_accuracy: 0.9120 - val_loss: 0.2360
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 71s 71ms/step - accuracy: 0.9496 - loss: 0.1326 - val_accuracy: 0.9120 - val_loss: 0.2633
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 40s 79ms/step - accuracy: 0.9650 - loss: 0.0941 - val_accuracy: 0.9180 - val_loss: 0.2343
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 45s 90ms/step - accuracy: 0.9716 - loss: 0.0732 - val_accuracy: 0.9115 - val_loss: 0.2585
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 36s 73ms/step - accuracy: 0.9806 - loss: 0.0514 - val_accuracy: 0.9105 - val_loss: 0.2854
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 37s 74ms/step - accuracy: 0.9816 - loss: 0.0522 - val_accuracy: 0.9065 - val_loss: 0.3032
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 39s 77ms/step - accuracy: 0.9862 - loss: 0.0370 - 

In [86]:
evaluate_model(bilstm_model,X_train,y_train,X_val,y_val,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.9934375

 Validition Accuracy :0.915

 Test Accuracy :0.905

 Classification Report
              precision    recall  f1-score   support

           0       0.88      0.90      0.89       275
           1       0.87      0.86      0.86       224
           2       0.92      0.94      0.93       695
           3       0.81      0.77      0.79       159
           4       0.95      0.96      0.95       581
           5       0.72      0.55      0.62        66

    accuracy                           0.91      2000
   macro avg       0.86      0.83      0.84      2000
weighted avg       0.90      0.91      0.90      2000


 Confusion Matrix
[[247   9   7   0  11   1]
 [  7 192   3   1  11  10]
 [  8   1 656  26   2   2]
 [  3   1  29 122   3   1]
 [ 11   3   9   1 557   0]
 [  4  15   9   0   2  36]]


### 6. Bi Directional GRU

In [63]:
bigru_model = Sequential([

    Embedding(vocab_size,
              embedding_dim,
              input_length=max_length),

    Bidirectional(
        GRU(128)
    ),

    Dense(num_classes,
          activation='softmax')
])

In [64]:
bigru_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [65]:
history_bilstm=bigru_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 36s 66ms/step - accuracy: 0.6145 - loss: 1.0159 - val_accuracy: 0.8460 - val_loss: 0.4393
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 67ms/step - accuracy: 0.9166 - loss: 0.2290 - val_accuracy: 0.9180 - val_loss: 0.2294
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 34s 69ms/step - accuracy: 0.9532 - loss: 0.1163 - val_accuracy: 0.9200 - val_loss: 0.2262
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 70ms/step - accuracy: 0.9710 - loss: 0.0782 - val_accuracy: 0.9260 - val_loss: 0.2080
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 71ms/step - accuracy: 0.9797 - loss: 0.0561 - val_accuracy: 0.9165 - val_loss: 0.2376
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 70ms/step - accuracy: 0.9829 - loss: 0.0430 - val_accuracy: 0.9150 - val_loss: 0.2564
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 70ms/step - accuracy: 0.9893 - loss: 0.0299 - val_accuracy: 0.9220 - val_loss: 0.2509
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 39s 78ms/step - accuracy: 0.9911 - loss: 0.0249 - 

In [87]:
evaluate_model(bigru_model,X_train,y_train,X_val,y_val,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.9955

 Validition Accuracy :0.9225

 Test Accuracy :0.9165

 Classification Report
              precision    recall  f1-score   support

           0       0.93      0.91      0.92       275
           1       0.87      0.90      0.89       224
           2       0.93      0.94      0.94       695
           3       0.78      0.82      0.80       159
           4       0.96      0.95      0.96       581
           5       0.82      0.62      0.71        66

    accuracy                           0.92      2000
   macro avg       0.88      0.86      0.87      2000
weighted avg       0.92      0.92      0.92      2000


 Confusion Matrix
[[251   5   5   2  12   0]
 [  4 202   2   2   7   7]
 [  3   2 656  31   1   2]
 [  1   0  28 130   0   0]
 [ 11   8   8   1 553   0]
 [  0  15   8   1   1  41]]


### 7.Stacked RNN

In [67]:
stacked_rnn = Sequential([

    Embedding(vocab_size,
              embedding_dim,
              input_length=max_length),

    SimpleRNN(
        128,
        return_sequences=True
    ),

    SimpleRNN(64),

    Dense(num_classes,
          activation='softmax')
])

In [68]:
stacked_rnn.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [69]:
history_stacked_rnn=stacked_rnn.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 30s 51ms/step - accuracy: 0.3279 - loss: 1.5870 - val_accuracy: 0.3520 - val_loss: 1.5840
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 25s 50ms/step - accuracy: 0.3272 - loss: 1.5831 - val_accuracy: 0.2745 - val_loss: 1.5915
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 25s 50ms/step - accuracy: 0.3266 - loss: 1.5805 - val_accuracy: 0.3520 - val_loss: 1.5852
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 26s 52ms/step - accuracy: 0.3276 - loss: 1.5806 - val_accuracy: 0.3520 - val_loss: 1.5917
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 25s 49ms/step - accuracy: 0.3259 - loss: 1.5816 - val_accuracy: 0.3520 - val_loss: 1.5803
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 24s 49ms/step - accuracy: 0.3262 - loss: 1.5809 - val_accuracy: 0.3520 - val_loss: 1.5863
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 25s 49ms/step - accuracy: 0.3219 - loss: 1.5822 - val_accuracy: 0.3520 - val_loss: 1.5936
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 25s 49ms/step - accuracy: 0.3278 - loss: 1.5814 - 

In [ ]:
evaluate_model(stacked_rnn,X_train,y_train,X_val,y_val,X_test,y_test)

### 8. Stacked LSTM

In [71]:
stacked_lstm = Sequential([

    Embedding(vocab_size,
              embedding_dim,
              input_length=max_length),

    LSTM(
        128,
        return_sequences=True
    ),

    LSTM(64),

    Dense(num_classes,
          activation='softmax')
])

In [72]:
stacked_lstm.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [73]:
history_stacked_lstm=stacked_lstm.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 47s 86ms/step - accuracy: 0.3278 - loss: 1.5820 - val_accuracy: 0.3520 - val_loss: 1.5839
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 45s 90ms/step - accuracy: 0.3285 - loss: 1.5779 - val_accuracy: 0.2750 - val_loss: 1.5910
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 45s 89ms/step - accuracy: 0.3322 - loss: 1.5756 - val_accuracy: 0.3515 - val_loss: 1.5829
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 45s 90ms/step - accuracy: 0.3327 - loss: 1.5782 - val_accuracy: 0.3480 - val_loss: 1.5852
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 44s 87ms/step - accuracy: 0.3336 - loss: 1.5761 - val_accuracy: 0.3520 - val_loss: 1.5794
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 44s 89ms/step - accuracy: 0.3338 - loss: 1.5743 - val_accuracy: 0.3520 - val_loss: 1.5796
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 44s 88ms/step - accuracy: 0.3327 - loss: 1.5747 - val_accuracy: 0.3520 - val_loss: 1.5821
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 44s 89ms/step - accuracy: 0.3332 - loss: 1.5761 - 

In [74]:
evaluate_model(stacked_lstm,X_train,y_train,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.4931875

 Test Accuracy :0.4775

 Classification Report
              precision    recall  f1-score   support

           0       0.67      0.01      0.01       275
           1       0.40      0.90      0.55       224
           2       0.52      0.92      0.66       695
           3       0.42      0.64      0.51       159
           4       0.71      0.02      0.04       581
           5       0.00      0.00      0.00        66

    accuracy                           0.48      2000
   macro avg       0.45      0.41      0.30      2000
weighted avg       0.55      0.48      0.35      2000


 Confusion Matrix
[[  2 227  10  36   0   0]
 [  0 201   7  16   0   0]
 [  0   3 639  48   5   0]
 [  1  17  40 101   0   0]
 [  0   8 536  25  12   0]
 [  0  45   7  14   0   0]]


In [75]:
### 9. Stacked GRU

In [76]:
stacked_gru = Sequential([

    Embedding(vocab_size,
              embedding_dim,
              input_length=max_length),

    GRU(
        128,
        return_sequences=True
    ),

    GRU(64),

    Dense(num_classes,
          activation='softmax')
])

In [77]:
stacked_gru.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [78]:
history_stacked_gru=stacked_gru.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 45s 81ms/step - accuracy: 0.3294 - loss: 1.5834 - val_accuracy: 0.2750 - val_loss: 1.6019
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 40s 80ms/step - accuracy: 0.3330 - loss: 1.5790 - val_accuracy: 0.3520 - val_loss: 1.5836
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 41s 82ms/step - accuracy: 0.5461 - loss: 1.1029 - val_accuracy: 0.8370 - val_loss: 0.4985
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 40s 80ms/step - accuracy: 0.9129 - loss: 0.2341 - val_accuracy: 0.9305 - val_loss: 0.1758
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 40s 81ms/step - accuracy: 0.9474 - loss: 0.1170 - val_accuracy: 0.9280 - val_loss: 0.1789
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 41s 82ms/step - accuracy: 0.9626 - loss: 0.0841 - val_accuracy: 0.9345 - val_loss: 0.1477
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 41s 81ms/step - accuracy: 0.9695 - loss: 0.0659 - val_accuracy: 0.9335 - val_loss: 0.1698
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 41s 82ms/step - accuracy: 0.9736 - loss: 0.0543 - 

In [79]:
evaluate_model(stacked_gru,X_train,y_train,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.9858125

 Test Accuracy :0.929

 Classification Report
              precision    recall  f1-score   support

           0       0.95      0.90      0.93       275
           1       0.82      0.97      0.89       224
           2       0.94      0.96      0.95       695
           3       0.84      0.84      0.84       159
           4       0.98      0.95      0.96       581
           5       0.95      0.64      0.76        66

    accuracy                           0.93      2000
   macro avg       0.91      0.88      0.89      2000
weighted avg       0.93      0.93      0.93      2000


 Confusion Matrix
[[247  19   2   0   7   0]
 [  0 218   1   0   5   0]
 [  1   2 667  23   0   2]
 [  0   0  26 133   0   0]
 [ 11  11   8   0 551   0]
 [  0  16   5   2   1  42]]
